# Objective measures for classification
In the following, typically used objective measures for classification tasks are introduced.

## Confusion matrix

A confusion matrix counts the detected words depending on the spoken/tested words. It is a compact logging tool for a classification task/experiment.
The spoken words corresponds to the rows of the matrix and the detected words corresponds to the columns of the matrix.

In the following experiment, a logging of a confusion matrix is simulated:

In [10]:
import numpy as np

NumberOfWordClasses = 4
NumberOfWordsPerClass = 100
AssumedAccuracy = 0.6
AssumedFalseRejectionRate = 0.05

NumberOfTotalWords = NumberOfWordClasses * NumberOfWordsPerClass
FalseRejectionCounter = 0
ConfusionMatrix = np.zeros((NumberOfWordClasses, NumberOfWordClasses))
for SpokenWordIndex in range(ConfusionMatrix.shape[0]):
    for n in range(NumberOfWordsPerClass):
        RandomNumber = np.random.rand(1)
        if RandomNumber < AssumedFalseRejectionRate:
            # false rejection occurs
            FalseRejectionCounter += 1
        elif RandomNumber < AssumedFalseRejectionRate + AssumedAccuracy:
            # correct classification occurs
            ConfusionMatrix[SpokenWordIndex, SpokenWordIndex] += 1
        else:
            # wrong classification occurs
            # find arbitrary wrong index
            TargetWrongIndex = SpokenWordIndex
            while TargetWrongIndex == SpokenWordIndex:
                TargetWrongIndex = np.random.randint(NumberOfWordClasses)
            ConfusionMatrix[SpokenWordIndex, TargetWrongIndex] += 1
            
print(ConfusionMatrix)

[[57. 14. 11. 13.]
 [10. 61.  9. 14.]
 [11. 12. 58. 13.]
 [ 9. 10. 11. 66.]]


There exists several parameters for the confusion matrix in order to evaluate the goodness of the underlying experiment of classification:

## Accuracy
Accuracy is the number of correct detections (the main diagonal) over the total sum of the confusion matrix. The range for the accuracy is 0..1. An accuracy of 0 is the worst case scenario with zero correct detected words. A scenario with an accuracy of 1 is a perfect scenario, where all spoken words are detected correctly.

In [11]:
NumberOfCorrectDetectedWords = 0
for SpokenWordIndex in range(ConfusionMatrix.shape[0]):
    NumberOfCorrectDetectedWords += ConfusionMatrix[SpokenWordIndex, SpokenWordIndex]
Accuracy = NumberOfCorrectDetectedWords / np.sum(ConfusionMatrix)
print('accuracy = ', str(Accuracy))

accuracy =  0.6385224274406333


## False rejections

Another parameter for evaluating a voice control algorithm is the false rejection rate. If a word is spoken and the voice detection algorithm ignores the spoken word, a false rejection occurs. The false rejection rate is the number of false rejections over the number of spoken/tested words.

In [12]:
NumberOfFalseRejections = NumberOfTotalWords - np.sum(ConfusionMatrix)
FalseRejectionRate = NumberOfFalseRejections / NumberOfTotalWords

print('Number of false rejections = ', str(int(NumberOfFalseRejections)))
print('false rejection rate = ', str(FalseRejectionRate))

Number of false rejections =  21
false rejection rate =  0.0525


## False alarms

False alarms occurs, if no word is spoken, but the voice control system detects a word due to background noise or internal errors. In a real voice control benchmark, the false alarms must be counted in order to evaluate the voice control algorithm correctly. In this artificial scenario with a random confusion matrix, the false alarms must be set to a random value.

In [13]:
NumberOfFalseAlarms = np.random.randint(NumberOfTotalWords * AssumedFalseRejectionRate * 2)
FalseAlarmRate = NumberOfFalseAlarms / NumberOfTotalWords

print('Number of false alarms = ', str(NumberOfFalseAlarms))
print('false alarm rate = ', str(FalseAlarmRate))

Number of false alarms =  19
false alarm rate =  0.0475


## Word error rate

In order to evaluate the overall performance of a classification task, the accuracy, the false alarms and the false rejections must be considered.

To simply compare two or more different classification algorithms, a single value measurement is beneficial. One example for such a single value measurement is the word error rate. It is defined as the sum of all errors (wrong classifications, false rejections and false alarms) over the number of spoken/testet words:

In [14]:
NumberOfWrongClassifications = np.sum(ConfusionMatrix) - NumberOfCorrectDetectedWords
NumberOfWordErrors = NumberOfFalseAlarms + NumberOfFalseRejections + NumberOfWrongClassifications
WordErrorRate = NumberOfWordErrors / NumberOfTotalWords

print('word error rate = ', str(WordErrorRate))

word error rate =  0.4425


## Precision

Precision is the fraction of the number of correct detections (the entry on the main diagonal) over the number of detections for this single word (the sum of the column).

In [15]:
for n in range(ConfusionMatrix.shape[1]):
    Precision = ConfusionMatrix[n, n] / np.sum(ConfusionMatrix[:, n])
    print('Precision for the ', n, '-th word: ', Precision)

Precision for the  0 -th word:  0.6551724137931034
Precision for the  1 -th word:  0.6288659793814433
Precision for the  2 -th word:  0.651685393258427
Precision for the  3 -th word:  0.6226415094339622


## Recall

Recall is the fraction of correct detections (the entry on the main diagonal) over the number of tests for this single word (the sum of the row):

In [16]:
for n in range(ConfusionMatrix.shape[0]):
    Recall = ConfusionMatrix[n, n] / np.sum(ConfusionMatrix[n, :])
    print('Recall for the ', n, '-th word: ', Recall)

Recall for the  0 -th word:  0.6
Recall for the  1 -th word:  0.648936170212766
Recall for the  2 -th word:  0.6170212765957447
Recall for the  3 -th word:  0.6875


## Accuracy paradox and Balanced trainingsdata
Accuracy is only a practical measure for goodness of fit for the training data if the training data is nearly balanced.

Balanced trainingsdata means, that nearly all classes to be detectet has nearly the same amount of training samples. For imbalanced trainingsdata, the classes has not the same probability in the trainingsdata.

A perfect dice will produce all six numbers nearly equally often. Therefore, each class ha a probability of roughly $\frac{1}{6}$ in the trainingsdata.

Trainingsdata for detecting earthquakes may be totally imbalanced, because in $99.99$ percent of all cases, there is no earthquake. Training with such a set of trainingsdata may result in the simplest possible classificator, which simply detects the class with the highest probability. In this case, the classificator always states: there is no earthquake. With this statement, an accuracy of $99.99$ percent is reached. But nevertheless, this classificator is useless.

The problem of a very high accuracy for a useless classificator is the accuracy paradox.

## F Score
In the case of unbalanced data, the F score is a better indicator for goodness of fit of the given classification compared to the accuracy.

The F score is the harmonic mean of the mean precision and the mean recall:

$F=\frac{2}{\frac{1}{\text{precision}}+\frac{1}{\text{recall}}}$

The bigger the F score, the better.